# Jupyter Notebook to clean data.xlsx

In [1]:
import pandas as pd
import numpy as np

## Loading the datasets

In [2]:
# replace with your Excel file path
file_path = "/Users/dazedinthecity/Documents/GitHub/CEU-DMDB-GroupH/data/private/data.xlsx"

# read first sheet into DataFrame
df = pd.read_excel(file_path, sheet_name=0)

# quick check
print(f"df shape: {df.shape}")
df.shape

df shape: (1846, 99)


(1846, 99)

In [3]:
# replace with your Excel file path
file_path_TE = "/Users/dazedinthecity/Desktop/Databases/TE_Course_Data_25-26.csv"

# read CSV into DataFrame
df_TE = pd.read_csv(file_path_TE)

# quick check
print(f"df_TE shape: {df_TE.shape}")
df_TE.shape

df_TE shape: (83, 93)


(83, 93)

## Step 1: Combine df and df_TE

In [4]:
# Show common and unique columns
common_cols = df.columns.intersection(df_TE.columns).tolist()
only_df = df.columns.difference(df_TE.columns).tolist()
only_df_TE = df_TE.columns.difference(df.columns).tolist()

print(f"Common columns ({len(common_cols)}): {common_cols}")
print(f"\nColumns only in df ({len(only_df)}): {only_df}")
print(f"\nColumns only in df_TE ({len(only_df_TE)}): {only_df_TE}")

# Concatenate (outer alignment of columns)
combined_df = pd.concat([df, df_TE], ignore_index=True, sort=False)
print(f"\nCombined dataframe shape: {combined_df.shape}")
print(f"Total rows: {len(df)} + {len(df_TE)} = {len(combined_df)}")
print(f"Total columns: {combined_df.shape[1]}")

# Quick peek
combined_df.head()

Common columns (92): ['Course code', 'Title', 'Offering periods', 'Offered', 'Taught', 'Allow repeats', 'US credits', 'Instructor', 'Level', 'Corresponding programs', 'Course occurence (code)', 'External ID', 'Abbreviation', 'Course type', 'Department', 'Organisation', 'Specification', 'Program (course owner)', 'Terminated', 'Start date', 'End date', 'Import date', 'Advised periods', 'Corresponding diet/collections codes', 'Corresponding diet/collections', 'Diet/collection descriptions', 'Diet/collection phases', 'Corresponding program codes', 'Resources', 'Subjects', 'Assessments', 'Exams (diff)', 'Exams (activity)', 'Course occurence', 'Course occurence (diff)', 'Course occurence (activity)', 'Temporary course code', 'Publication date', 'CIVICA', 'OSUN', 'Academic writing course', 'Assessment type', 'Offered only for department(s)', 'Course offered for cross-listing', 'Offered only for program(s)', 'Historic code', 'Historic title', 'Marking scheme', 'Methods course', 'Offered for no

/var/folders/ty/1b7rrh3s0yx8wvyxm5v17jdw0000gp/T/ipykernel_15607/3782681540.py:11: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined_df = pd.concat([df, df_TE], ignore_index=True, sort=False)


,Course code,Title,Offering periods,Offered,Taught,Allow repeats,US credits,Instructor,Level,Corresponding programs,...,Load division (count),Course late changes (Winter term),Course late changes (Spring term),Course late changes,Course details,Course detail late changes,Manage course basics,New course,Instructor (changes),Load division
0,ECBS6011,Advanced Econometrics 1,Fall term,1.0,Yes,No,2.0,Adam Reiff (reiffa@ceu.edu),Doctoral or equivalent,"PhD in Economics, MA in Economics, Data, and P...",...,1,Change applied,NaN,NaN,Maintain,Change applied,Approved,NaN,False,NaN
1,ECBS6255,Theoretical Econometrics,Fall term,1.0,Yes,No,2.0,Adam Reiff (reiffa@ceu.edu),Doctoral or equivalent,"MA in Economics, PhD in Economics, MS in Socia...",...,1,Change applied,NaN,NaN,Maintain,Change course details,Approved,NaN,False,NaN
2,GENS5070,Internship Analysis Workshop,Fall term,1.0,Yes,No,2.0,Adriana Qubaiova (qubaiovaa@ceu.edu),Master or equivalent,MA in Critical Gender Studies,...,1,Change applied,NaN,NaN,NaN,Change course details,Approved,NaN,True,NaN
3,GENS5119,Academic Writing Part 2: Thesis Development,Winter term,1.0,Yes,No,1.0,Adriana Qubaiova (qubaiovaa@ceu.edu),Master or equivalent,"MA in Women's and Gender Studies (GEMMA), MA i...",...,1,NaN,NaN,NaN,Approved,Change applied,Approved,NaN,True,NaN
4,GENS5564,Gender and Sexuality in West Asia & North Africa,Fall term,1.0,Yes,No,4.0,Adriana Qubaiova (qubaiovaa@ceu.edu),Master or equivalent,"PhD in Comparative Gender Studies, MA in Natio...",...,1,Change applied,NaN,NaN,Approved,Change applied,Approved,NaN,True,NaN


## Step 2: Remove rows with NAs >= 20%

In [5]:
# Calculate the percentage of NAs for each row
na_threshold_row = 0.50  # 20%
total_cols = combined_df.shape[1]

# Count NAs per row
na_counts_per_row = combined_df.isna().sum(axis=1)
na_pct_per_row = na_counts_per_row / total_cols

# Identify rows to keep (those with < 20% NAs)
rows_to_keep = na_pct_per_row < na_threshold_row

print(f"Original number of rows: {len(combined_df)}")
print(f"Rows with >= 20% NAs: {(~rows_to_keep).sum()}")
print(f"Rows with < 20% NAs (to keep): {rows_to_keep.sum()}")

# Filter the dataframe
df_cleaned_rows = combined_df[rows_to_keep].copy()

print(f"\nDataframe shape after row cleaning: {df_cleaned_rows.shape}")
print(f"Rows removed: {len(combined_df) - len(df_cleaned_rows)}")

Original number of rows: 1929
Rows with >= 20% NAs: 1329
Rows with < 20% NAs (to keep): 600

Dataframe shape after row cleaning: (600, 100)
Rows removed: 1329


## Step 3: Remove columns with NAs >= 50%

In [6]:
# Calculate the percentage of NAs for each column
na_threshold_col = 0.50  # 50%
total_rows = df_cleaned_rows.shape[0]

# Count NAs per column
na_counts_per_col = df_cleaned_rows.isna().sum()
na_pct_per_col = na_counts_per_col / total_rows

# Identify columns to keep (those with < 50% NAs)
cols_to_keep = na_pct_per_col < na_threshold_col
cols_to_drop = na_pct_per_col >= na_threshold_col

print(f"Original number of columns: {len(df_cleaned_rows.columns)}")
print(f"Columns with >= 50% NAs: {cols_to_drop.sum()}")
print(f"Columns with < 50% NAs (to keep): {cols_to_keep.sum()}")

# Show which columns will be dropped
if cols_to_drop.sum() > 0:
    print(f"\nColumns to be dropped ({cols_to_drop.sum()}):")
    for col in df_cleaned_rows.columns[cols_to_drop]:
        pct = na_pct_per_col[col] * 100
        print(f"  - {col}: {pct:.1f}% NAs")

# Filter the dataframe
df_final = df_cleaned_rows.loc[:, cols_to_keep].copy()

print(f"\nFinal dataframe shape: {df_final.shape}")
print(f"Columns removed: {len(df_cleaned_rows.columns) - len(df_final.columns)}")

Original number of columns: 100
Columns with >= 50% NAs: 45
Columns with < 50% NAs (to keep): 55

Columns to be dropped (45):
  - Original code: 96.2% NAs
  - Name: 100.0% NAs
  - Organisation: 100.0% NAs
  - Specification: 100.0% NAs
  - Program (course owner): 70.2% NAs
  - Contact hours: 100.0% NAs
  - Start date: 100.0% NAs
  - End date: 100.0% NAs
  - Advised periods: 100.0% NAs
  - Diet/collection descriptions: 100.0% NAs
  - Diet/collection phases: 100.0% NAs
  - Resources: 100.0% NAs
  - Subjects: 98.8% NAs
  - Assessments: 100.0% NAs
  - Exams (diff): 100.0% NAs
  - Exams (activity): 100.0% NAs
  - Course occurence (activity): 100.0% NAs
  - CIVICA: 98.5% NAs
  - OSUN: 98.3% NAs
  - Academic writing course: 96.3% NAs
  - Offered only for department(s): 62.8% NAs
  - Offered only for program(s): 83.0% NAs
  - Historic code: 100.0% NAs
  - Historic title: 100.0% NAs
  - Methods course: 97.3% NAs
  - Find your own course: 96.7% NAs
  - Please revive my course: 94.8% NAs
  - Sub t

## Summary of data cleaning

In [7]:
print("="*60)
print("DATA CLEANING SUMMARY")
print("="*60)
print(f"\n1. Combined datasets:")
print(f"   - df: {df.shape}")
print(f"   - df_TE: {df_TE.shape}")
print(f"   - Combined: {combined_df.shape}")

print(f"\n2. Row cleaning (removed rows with >= 20% NAs):")
print(f"   - Before: {len(combined_df)} rows")
print(f"   - After: {len(df_cleaned_rows)} rows")
print(f"   - Removed: {len(combined_df) - len(df_cleaned_rows)} rows")

print(f"\n3. Column cleaning (removed columns with >= 50% NAs):")
print(f"   - Before: {len(df_cleaned_rows.columns)} columns")
print(f"   - After: {len(df_final.columns)} columns")
print(f"   - Removed: {len(df_cleaned_rows.columns) - len(df_final.columns)} columns")

print(f"\n4. Final dataset:")
print(f"   - Shape: {df_final.shape}")
print(f"   - Total cells: {df_final.shape[0] * df_final.shape[1]:,}")
print(f"   - Total NAs: {df_final.isna().sum().sum():,}")
print(f"   - Overall NA percentage: {(df_final.isna().sum().sum() / (df_final.shape[0] * df_final.shape[1]) * 100):.2f}%")
print("="*60)

DATA CLEANING SUMMARY

1. Combined datasets:
   - df: (1846, 99)
   - df_TE: (83, 93)
   - Combined: (1929, 100)

2. Row cleaning (removed rows with >= 20% NAs):
   - Before: 1929 rows
   - After: 600 rows
   - Removed: 1329 rows

3. Column cleaning (removed columns with >= 50% NAs):
   - Before: 100 columns
   - After: 55 columns
   - Removed: 45 columns

4. Final dataset:
   - Shape: (600, 55)
   - Total cells: 33,000
   - Total NAs: 1,744
   - Overall NA percentage: 5.28%


## Preview of final cleaned data

In [8]:
# Display first few rows
df_final.head()

,Course code,Title,Offering periods,Offered,Taught,Allow repeats,US credits,Instructor,Level,Corresponding programs,...,Changes in Course prerequisites (EN),Changes in Waiting list handling (EN),Instructor (id),Instructor (email),Load division (count),Course late changes (Winter term),Course details,Course detail late changes,Manage course basics,Instructor (changes)
0,ECBS6011,Advanced Econometrics 1,Fall term,1.0,Yes,No,2.0,Adam Reiff (reiffa@ceu.edu),Doctoral or equivalent,"PhD in Economics, MA in Economics, Data, and P...",...,True,False,reiffa@ceu.edu,reiffa@ceu.edu,1,Change applied,Maintain,Change applied,Approved,False
1,ECBS6255,Theoretical Econometrics,Fall term,1.0,Yes,No,2.0,Adam Reiff (reiffa@ceu.edu),Doctoral or equivalent,"MA in Economics, PhD in Economics, MS in Socia...",...,True,False,reiffa@ceu.edu,reiffa@ceu.edu,1,Change applied,Maintain,Change course details,Approved,False
3,GENS5119,Academic Writing Part 2: Thesis Development,Winter term,1.0,Yes,No,1.0,Adriana Qubaiova (qubaiovaa@ceu.edu),Master or equivalent,"MA in Women's and Gender Studies (GEMMA), MA i...",...,True,False,qubaiovaa@ceu.edu,qubaiovaa@ceu.edu,1,NaN,Approved,Change applied,Approved,True
4,GENS5564,Gender and Sexuality in West Asia & North Africa,Fall term,1.0,Yes,No,4.0,Adriana Qubaiova (qubaiovaa@ceu.edu),Master or equivalent,"PhD in Comparative Gender Studies, MA in Natio...",...,True,False,qubaiovaa@ceu.edu,qubaiovaa@ceu.edu,1,Change applied,Approved,Change applied,Approved,True
6,GENS5113,Feminist Qualitative Research Methods,Fall term,1.0,Yes,No,2.0,Adriana Qubaiova (qubaiovaa@ceu.edu),Master or equivalent,"MA in Gender Studies, MA in Women's and Gender...",...,True,False,qubaiovaa@ceu.edu,qubaiovaa@ceu.edu,1,NaN,Approved,Change applied,Approved,True


In [9]:
# Display info about the final dataframe
df_final.info()

<class 'pandas.core.frame.DataFrame'>
Index: 600 entries, 0 to 1926
Data columns (total 55 columns):
 #   Column                                                    Non-Null Count  Dtype         
---  ------                                                    --------------  -----         
 0   Course code                                               600 non-null    object        
 1   Title                                                     600 non-null    object        
 2   Offering periods                                          594 non-null    object        
 3   Offered                                                   600 non-null    float64       
 4   Taught                                                    600 non-null    object        
 5   Allow repeats                                             579 non-null    object        
 6   US credits                                                600 non-null    float64       
 7   Instructor                                      

In [22]:
columns_to_drop = [
    'Taught',
    'Course occurence (code)',
    'UID',
    'External ID',
    'Terminated',
    'Ignore in TE Curriculum',
    'Import date','Course occurence',
    'Course occurence (diff)',
    'Temporary course code',
    'Publication date',
    'Assessment type',
    'Course offered for cross-listing',
    'Changes in Learning outcomes (EN)',
    'Changes in Learning activities and teaching methods (EN)',
    'Changes in Assessment (EN)',
    'Changes in Course contents (EN)',
    'Changes in Background and overall aim (EN)',
    'Changes in Contact details (EN)',
    'Changes in Course prerequisites (EN)',
    'Changes in Waiting list handling (EN)',
    'Load division (count)',
    'Course late changes (Winter term)',
    'Course detail late changes',
    'Course details',
    'Manage course basics',
    'Instructor (changes)'
    ]
cols_found = [col for col in columns_to_drop if col in df_final.columns]

if cols_found:
    df_final = df_final.drop(columns=cols_found)
    print(f"Dropped columns: {cols_found}")
    print(f"New shape: {df_final.shape}")
else:
    print(f"Columns not found in df_final")
    print(f"Available columns: {df_final.columns.tolist()}")

Dropped columns: ['Taught', 'Course occurence (code)', 'External ID', 'Terminated', 'Ignore in TE Curriculum', 'Import date', 'Course occurence', 'Course occurence (diff)', 'Temporary course code', 'Publication date', 'Assessment type', 'Course offered for cross-listing', 'Changes in Learning outcomes (EN)', 'Changes in Learning activities and teaching methods (EN)', 'Changes in Assessment (EN)', 'Changes in Course contents (EN)', 'Changes in Background and overall aim (EN)', 'Changes in Contact details (EN)', 'Changes in Course prerequisites (EN)', 'Changes in Waiting list handling (EN)', 'Load division (count)', 'Course late changes (Winter term)', 'Course detail late changes', 'Course details', 'Manage course basics', 'Instructor (changes)']
New shape: (600, 28)


## Export cleaned data (optional)

In [23]:
# Uncomment to save the cleaned data
output_path = "cleaned_data.xlsx"
df_final.to_excel(output_path, index=False)
print(f"Cleaned data saved to: {output_path}")



Cleaned data saved to: cleaned_data.xlsx
